# Image-text alignment without a checkpoint

The local exercise uses two small towers and hand-built feature rows to make normalization and symmetric contrastive loss observable.

In [ ]:
from pathlib import Path
import importlib.util
import sys
lesson_rel = Path('phases/04-computer-vision/18-open-vocab-clip')
candidates = []
for base in (Path.cwd(), *Path.cwd().parents):
    candidates.extend((base / lesson_rel / 'code/main.py', base / 'code/main.py'))
code_path = next(p.resolve() for p in candidates if p.is_file())
spec = importlib.util.spec_from_file_location('cv04_l18_nb', code_path)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
assert spec.loader is not None
spec.loader.exec_module(module)
print(code_path)

In [ ]:
if not module.TORCH_AVAILABLE:
    import numpy as np
    image = text = np.eye(3)
    loss = module.numpy_clip_loss(image, text, 2.0)
    labels = module.numpy_zero_shot_classify(image, text, ['red', 'blue', 'green'])
    assert np.isfinite(loss) and labels == ['red', 'blue', 'green']
    print({'Build-It': 'NumPy two-tower seams', 'loss': loss, 'labels': labels, 'Use-It': 'PyTorch skipped cleanly'})
else:
    import torch
    model = module.TwoTower(img_in=3, txt_in=3, emb=3)
    image = torch.eye(3)
    text = torch.eye(3)
    image_z, text_z, scale = model(image, text)
    loss = module.clip_loss(image_z, text_z, scale)
    assert image_z.shape == text_z.shape == (3, 3) and torch.isfinite(loss)
    print({'loss': float(loss), 'scale': float(scale)})

Zero-shot labels are simply the argmax over image-to-class-text similarity here; model quality and vocabulary coverage are outside the fixture.